# SO-101 reach VLA: training on Colab

Fine-tunes **Qwen3.5-2B (LoRA, Unsloth)** on the reach dataset from `so101_real/reach_dataset.py`, the same training as
`so101_real/real_train.py` on the laptop. Self-contained: only the dataset zip on Google Drive is needed.

**Before running:** *Runtime → Change runtime type → GPU* (A100 or L4 if available; T4 works but is slow).

Run the cells top to bottom. Checkpoints are written to Drive every 500 steps; if Colab disconnects, run all cells again
and training **continues from the last checkpoint**.

### 1. Settings: edit these

In [ ]:
# Where the dataset zip is on your Google Drive
DATA_ZIP = "/content/drive/MyDrive/so101_reach_v1_26-10-2026-13-21-29.zip"
# Where checkpoints and the final LoRA go (on Drive, so nothing is lost when Colab disconnects)
RUN_DIR = "/content/drive/MyDrive/so101_vla_runs/qwen35_2b_so101_reach_colab"

EPOCHS = 1
MAX_STEPS = -1        # e.g. 30 for a quick smoke test; -1 = full epochs
SAVE_STEPS = 500

# Laptop path of the same dataset: written into the run's config.json so real_robot_policy.py recognises the task
LOCAL_DATASETS = "/home/asus/Desktop/works/60_pick_place_advanced/so101_real/datasets"

### 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 3. Install (same versions as the laptop)
If a later cell fails with an import / version error: *Runtime → Restart session*, then run from step 4 on (no need to reinstall).

In [ ]:
%%capture
!pip install -q "unsloth==2026.10.7" "unsloth_zoo==2026.10.7" "transformers==5.17.0" "trl==1.15.0" "peft==0.21.2" bitsandbytes

### 4. Mount Drive and unzip the dataset to the fast local disk

In [ ]:
import os, glob, json
from google.colab import drive
drive.mount("/content/drive")

assert os.path.exists(DATA_ZIP), f"zip not found: {DATA_ZIP}"
if not glob.glob("/content/data/**/train.jsonl", recursive=True):
    import subprocess
    os.makedirs("/content/data", exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", DATA_ZIP, "-d", "/content/data"], check=True)
DATASET_DIR = os.path.dirname(glob.glob("/content/data/**/train.jsonl", recursive=True)[0])
meta = json.load(open(os.path.join(DATASET_DIR, "meta.json")))
n_train = sum(1 for _ in open(os.path.join(DATASET_DIR, "train.jsonl")))
print("dataset :", DATASET_DIR)
print("task    :", meta.get("task"), "| phases", meta.get("phases"), "| cameras", meta.get("cameras"))
print("samples :", n_train, "train | top camera calibrated:", meta.get("top_camera_calibrated"))

### 5. Dataset loader
Same as `vlm_training/pick_place_dataset.py`: samples in the Unsloth chat format, images opened lazily.

In [ ]:
from PIL import Image

class Conversations:
    def __init__(self, root, split="train"):
        self.root = root
        with open(os.path.join(root, f"{split}.jsonl")) as f:
            self.rows = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        content = [{"type": "image", "image": Image.open(os.path.join(self.root, p)).convert("RGB")}
                   for p in row["images"]]
        content.append({"type": "text", "text": row["prompt"]})
        return {"messages": [{"role": "user", "content": content},
                             {"role": "assistant", "content": [{"type": "text", "text": row["answer"]}]}]}

    def to_hf_dataset(self):
        from datasets import Dataset   # trl >= 1.x needs a datasets.Dataset
        return Dataset.from_dict({"idx": list(range(len(self)))}).with_transform(
            lambda batch: {"messages": [self[i]["messages"] for i in batch["idx"]]})

train_dataset = Conversations(DATASET_DIR, "train").to_hf_dataset()
print(len(train_dataset), "train samples; example answer:", Conversations(DATASET_DIR)[0]["messages"][1]["content"][0]["text"])

### 6. Model + LoRA
`UNSLOTH_BLOCK_COMPILE=0` is the same setting the laptop needs: Qwen3.5's linear-attention layers otherwise compile
for ages. It has to be set before `unsloth` is imported.

In [ ]:
os.environ["UNSLOTH_BLOCK_COMPILE"] = "0"
import unsloth  # before transformers / trl
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3.5-2B", load_in_4bit=False,
                                                   use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

### 7. Train
The effective batch stays 8 (like the laptop) and only the split changes with GPU memory: bigger GPUs run more samples
at once and accumulate less, which is faster for the same result.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
batch, accum = (8, 1) if gpu_gb >= 35 else (4, 2) if gpu_gb >= 20 else (2, 4)
print(f"GPU {torch.cuda.get_device_name(0)} {gpu_gb:.0f} GB -> batch {batch} x accumulation {accum}")

os.makedirs(RUN_DIR, exist_ok=True)
FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=batch, gradient_accumulation_steps=accum,
        warmup_steps=50, num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
        learning_rate=2e-4, logging_steps=10, save_steps=SAVE_STEPS, save_total_limit=4,
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear", seed=3407,
        output_dir=os.path.join(RUN_DIR, "trainer"), report_to="none",
        remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048),
)
resume = bool(glob.glob(os.path.join(RUN_DIR, "trainer", "checkpoint-*")))
print("resuming from the last checkpoint" if resume else "starting a new run")
stats = trainer.train(resume_from_checkpoint=True if resume else None)
print(f"done in {stats.metrics['train_runtime'] / 60:.1f} min, final loss {stats.metrics.get('train_loss', float('nan')):.4f}")

### 8. Save the LoRA (+ a zip to download)
`config.json` points at the laptop copy of the dataset, so `real_robot_policy.py` knows this is a **reach** model.

In [ ]:
import shutil
from datetime import datetime

lora_dir = os.path.join(RUN_DIR, "lora")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
run_name = os.path.basename(RUN_DIR.rstrip("/"))
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    json.dump({"args": {"dataset": os.path.join(LOCAL_DATASETS, os.path.basename(DATASET_DIR)), "model": "unsloth/Qwen3.5-2B",
                        "epochs": EPOCHS, "max_steps": MAX_STEPS, "batch": batch, "grad_accum": accum},
               "trained_on": "colab", "finished": datetime.now().isoformat(timespec="seconds")}, f, indent=2)

# zip = <run>/lora + <run>/config.json (without the big trainer/ checkpoints)
stage = f"/content/export/{run_name}"
shutil.rmtree("/content/export", ignore_errors=True)
shutil.copytree(lora_dir, os.path.join(stage, "lora"))
shutil.copy(os.path.join(RUN_DIR, "config.json"), stage)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(RUN_DIR), run_name + "_lora"), "zip", "/content/export", run_name)
print("saved:", lora_dir)
print("download this zip from Drive:", zip_path)

### 9. Back on the laptop
1. Download `<run>_lora.zip` from Drive (`so101_vla_runs/`).
2. Unzip it into `so101_real/checkpoints/`, which gives `so101_real/checkpoints/qwen35_2b_so101_reach_colab/lora/`.
3. Test in the sim, then on the real arm:
```bash
cd ~/Desktop/works/60_pick_place_advanced/so101_real
python reach_policy.py --lora checkpoints/qwen35_2b_so101_reach_colab/lora --episodes 20 --headless
python real_robot_policy.py --lora checkpoints/qwen35_2b_so101_reach_colab/lora --no-arm
```